# California Multifamily Housing APR Explorer

Run All loads and validates the archived HCD APR 2018–2024 release. It never prepares source data or fits models. Model publication is a separate, owner-only manual release workflow.

Local dry-run (fixture release for notebook development):

```bash
python3 scripts/export_pages_catalog.py --fixture --staging-dir docs/data/releases/2018-2024
python3 scripts/verify_pages_catalog.py docs/data/releases/2018-2024
```


In [ ]:
import json
from pathlib import Path

repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'docs' / 'index.html').exists()), None)
if repo is None:
    raise FileNotFoundError('Run from the APR Explorer repository')
release = repo / 'docs' / 'data' / 'releases' / '2018-2024'
required = ['manifest.json', 'chart_labels.json', 'catalog.json', 'map_metrics.json', 'maps.geojson']
missing = [name for name in required if not (release / name).exists()]
if missing:
    raise FileNotFoundError(
        f'Archived release incomplete: {missing}. Run the fixture dry-run in the intro cell.'
    )
artifact_keys = {
    'manifest.json': 'manifest',
    'chart_labels.json': 'chart_labels',
    'catalog.json': 'catalog',
    'map_metrics.json': 'map_metrics',
    'maps.geojson': 'maps',
}
artifacts = {artifact_keys[name]: json.loads((release / name).read_text()) for name in required}
if artifacts['manifest'].get('release_id') != '2018-2024':
    raise ValueError('Unexpected release id')
if not artifacts['catalog'] or not artifacts['map_metrics'] or not artifacts['maps'].get('features'):
    raise ValueError('Malformed archived release')
print(
    f"Release {artifacts['manifest']['release_id']} built {artifacts['manifest'].get('built_at', 'unknown')}"
)


In [ ]:
pairs = [
    dict(zip(('geography', 'y_col', 'x_col', 'robustness'), key.split(':')), payload=payload)
    for key, payload in artifacts['catalog'].items()
]
missing_predictors = sorted({p['x_col'] for p in pairs} - set(artifacts['chart_labels']['predictors']))
missing_outcomes = sorted({p['y_col'] for p in pairs} - set(artifacts['chart_labels']['outcomes']))
print({'missing_predictors': missing_predictors, 'missing_outcomes': missing_outcomes})


## Maps


In [ ]:
import math
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

geo_views = {
    'Incorporated cities': {'city'},
    'Whole counties': {'county_whole'},
    'Cities + unincorporated county': {'city', 'county_residual'},
}
map_subheaders = {
    'Incorporated cities': 'Incorporated city jurisdictions',
    'Whole counties': 'Whole county jurisdictions',
    'Cities + unincorporated county': 'Incorporated cities and unincorporated county jurisdictions',
}
geography_view = widgets.Dropdown(
    description='Geography view',
    options=list(geo_views),
    value='Cities + unincorporated county',
)
def map_metric_label(metric):
    y_col = metric.get('y_col')
    if y_col:
        return artifacts['chart_labels']['outcomes'].get(y_col, metric.get('title', y_col))
    return metric.get('title', metric.get('key', ''))

metric_by_title = {map_metric_label(m): m for m in artifacts['map_metrics']}
map_metric = widgets.Dropdown(description='Map metric', options=list(metric_by_title))
map_subheader = widgets.HTML()
map_unit_hint = widgets.HTML()
map_output = widgets.Output()


def finite_values(values):
    return [float(v) for v in values if isinstance(v, (int, float)) and math.isfinite(v)]


def draw_map(*_):
    metric = metric_by_title[map_metric.value]
    features = [
        f
        for f in artifacts['maps']['features']
        if f['properties']['geo_type'] in geo_views[geography_view.value]
    ]
    geojson = {'type': 'FeatureCollection', 'features': features}
    z_values = [f['properties'].get(metric['metric_col']) for f in features]
    div = metric['cmap_kind'] == 'div'
    trace_kwargs = {}
    values = finite_values(z_values)
    if values:
        if div:
            bound = max(abs(v) for v in values)
            trace_kwargs.update(zmid=0, zmin=-bound, zmax=bound)
        else:
            trace_kwargs.update(zmin=min(values), zmax=max(values))
    trace = go.Choroplethmapbox(
        geojson=geojson,
        locations=[f['properties']['feature_id'] for f in features],
        z=z_values,
        featureidkey='properties.feature_id',
        colorscale='RdYlGn' if div else 'PuRd',
        below='water',
        marker={'opacity': 0.78, 'line': {'color': 'rgba(255,255,255,.72)', 'width': 0.45}},
        text=[f['properties'].get('city_name') or f['properties'].get('county_name') for f in features],
        hovertemplate='%{text}<br>%{z}<extra></extra>',
        **trace_kwargs,
    )
    unit = ' (per 1,000 population)' if metric.get('unit') == 'per_1000_pop' else ''
    map_subheader.value = f"<p><em>{map_subheaders[geography_view.value]}</em></p>"
    map_unit_hint.value = f"<strong>{unit}</strong>" if unit else ''
    with map_output:
        map_output.clear_output(wait=True)
        go.Figure(trace).update_layout(
            mapbox_style='carto-positron',
            mapbox_zoom=4.5,
            mapbox_center={'lat': 37.2, 'lon': -119.5},
            dragmode='zoom',
            uirevision=f"{geography_view.value}:{metric['key']}",
            title=map_metric_label(metric),
        ).show(config={'scrollZoom': True})


geography_view.observe(draw_map, names='value')
map_metric.observe(draw_map, names='value')
display(widgets.HBox([geography_view, map_metric, map_unit_hint]), map_subheader, map_output)
draw_map()


## Models


In [ ]:
import pandas as pd

GEO_NAMES = {'city': 'City', 'zip': 'ZIP code'}
MODEL_LABELS = {
    'stationary_bootstrap': 'Two-Part MLE + Stationary Bootstrap',
    'hierarchical': 'Hierarchical Bayes',
    'both': 'Both',
}
MODEL_LEGEND = {
    'x': 0.02,
    'y': 0.98,
    'xanchor': 'left',
    'yanchor': 'top',
    'bgcolor': 'rgba(255,255,255,.78)',
    'bordercolor': '#ddd',
    'borderwidth': 1,
}
chart_labels = artifacts['chart_labels']
geography = widgets.Dropdown(description='Geography')
x_col = widgets.Dropdown(description='Variable (X)')
y_col = widgets.Dropdown(description='Variable (Y)')
model_display = widgets.Dropdown(description='Model display')
zero_values = widgets.Dropdown(
    description='Zero Values',
    options=['Two-Part Hurdle', 'Positive Only'],
    value='Two-Part Hurdle',
)
robustness = widgets.Dropdown(description='Robustness')
chart_output = widgets.Output()
stats_output = widgets.Output()
entries = [
    dict(zip(('geography', 'y_col', 'x_col', 'robustness'), key.split(':')), payload=payload)
    for key, payload in artifacts['catalog'].items()
]


def label_for(name, value):
    if name == 'geography':
        return GEO_NAMES.get(value, value)
    if name in {'x_col', 'y_col'}:
        return chart_labels['outcomes'].get(value, chart_labels['predictors'].get(value, value))
    return value


def set_options(control, name, values, preferred=None):
    current = control.value
    ordered = sorted(values)
    control.options = [(label_for(name, value), value) for value in ordered]
    target = preferred if preferred is not None else current
    control.value = target if target in ordered else (ordered[0] if ordered else None)


def outcomes_for_geo(geo):
    suffix = '_CO_total' if geo == 'city' else '_CO'
    return [key for key in chart_labels['outcomes'] if key.endswith(suffix)]


def predictors_for_geo(geo):
    return [
        key
        for key in chart_labels.get('predictorApplicability', {}).get(geo, chart_labels['predictors'])
        if key in chart_labels['predictors']
    ]


def variables_for_geo(geo):
    seen = set()
    values = []
    for key in [*outcomes_for_geo(geo), *predictors_for_geo(geo)]:
        if key not in seen:
            seen.add(key)
            values.append(key)
    return values


def robustness_for_geo(geo):
    return {p['robustness'] for p in entries if p['geography'] == geo}


def selected_key():
    return f"{geography.value}:{y_col.value}:{x_col.value}:{robustness.value}"


def settle():
    set_options(geography, 'geography', {p['geography'] for p in entries})
    variables = variables_for_geo(geography.value)
    fallback = next((p for p in entries if p['geography'] == geography.value and p['y_col'] != p['x_col']), None)
    preferred_x = x_col.value if x_col.value in variables else (fallback['x_col'] if fallback else (variables[0] if variables else None))
    preferred_y = y_col.value if y_col.value in variables and y_col.value != preferred_x else (
        fallback['y_col'] if fallback and fallback['y_col'] != preferred_x else next((v for v in variables if v != preferred_x), None)
    )
    set_options(x_col, 'x_col', [v for v in variables if v != preferred_y], preferred_x)
    set_options(y_col, 'y_col', [v for v in variables if v != x_col.value], preferred_y)
    set_options(robustness, 'robustness', robustness_for_geo(geography.value))
    pair = artifacts['catalog'].get(selected_key())
    avail = (pair or {}).get('availability', {})
    boot_ok = bool(avail.get('stationary_bootstrap'))
    hier_ok = bool(avail.get('hierarchical'))
    choices = (
        (['both'] if boot_ok and hier_ok else [])
        + (['stationary_bootstrap'] if boot_ok else [])
        + (['hierarchical'] if hier_ok else [])
    )
    old = model_display.value
    model_display.options = [(MODEL_LABELS[value], value) for value in choices]
    model_display.value = old if old in choices else (choices[0] if choices else None)
    return pair


def add_band(fig, pair, component, name, color):
    if not component or 'lower' not in component or 'upper' not in component:
        return
    x = pair['x_grid']
    fig.add_scatter(x=x, y=component['lower'], mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip')
    fig.add_scatter(
        x=x,
        y=component['upper'],
        mode='lines',
        line={'width': 0},
        fill='tonexty',
        fillcolor=color.replace('1)', '0.18)'),
        name=name,
        hoverinfo='skip',
    )


def add_mean(fig, pair, component, name, color):
    if component and 'mean' in component:
        fig.add_scatter(x=pair['x_grid'], y=component['mean'], mode='lines', line={'color': color}, name=name)


def diagnostics_table(pair):
    stats = pair['stats']['two_part']
    return pd.DataFrame(
        [
            {'Part': 'Zero (logit)', 'Parameter': 'α', 'Coefficient': stats['alpha'], 't': None, 'p': None},
            {
                'Part': 'Zero (logit)',
                'Parameter': 'β',
                'Coefficient': stats['beta'],
                't': stats.get('beta_t'),
                'p': stats.get('beta_p'),
            },
            {'Part': 'Positive', 'Parameter': 'γ', 'Coefficient': stats['intercept'], 't': None, 'p': None},
            {
                'Part': 'Positive',
                'Parameter': 'δ',
                'Coefficient': stats['slope'],
                't': stats.get('slope_t'),
                'p': stats.get('slope_p'),
            },
        ]
    )


def y_axis_title(pair):
    title = label_for('y_col', pair['y_col'])
    if pair['y_col'] in chart_labels.get('per1000Outcomes', []):
        return f'{title} per 1,000 pop'
    return title


def view_key_for_pair(pair):
    # Continuous (econ-Y) pairs have no zero/hurdle part — always render positive_only,
    # regardless of the Zero Values control (mirrors docs/index.html::viewKeyForPair).
    if pair.get('model_family') == 'continuous':
        return 'positive_only'
    return 'two_part_hurdle' if zero_values.value == 'Two-Part Hurdle' else 'positive_only'


def draw_model(*_):
    pair = settle()
    if pair is None:
        with chart_output:
            chart_output.clear_output(wait=True)
            print('This combination was not exported (insufficient jurisdictions or missing yearly data). Choose another pair.')
        with stats_output:
            stats_output.clear_output(wait=True)
        return
    view_key = view_key_for_pair(pair)
    view = pair['views'][view_key]
    obs = pair['observations']
    keep_zeros = view_key == 'two_part_hurdle'
    keep = [i for i, y in enumerate(obs['y']) if keep_zeros or y > 0]
    fig = go.Figure(
        go.Scatter(
            x=[obs['x'][i] for i in keep],
            y=[obs['y'][i] for i in keep],
            text=[obs['labels'][i] for i in keep],
            mode='markers',
            name='Observations',
            marker={'color': ['#777' if obs['y'][i] == 0 else '#222' for i in keep]},
            hovertemplate='%{text}<br>x=%{x}<br>y=%{y}<extra></extra>',
        )
    )
    selected = model_display.value
    if selected != 'hierarchical':
        add_band(fig, pair, view.get('stationary_bootstrap'), 'Stationary bootstrap 95% interval', 'rgba(0,151,167,1)')
        add_mean(fig, pair, view.get('mle'), 'Two-part MLE', 'rgba(0,151,167,1)')
    if selected != 'stationary_bootstrap':
        add_band(fig, pair, view.get('hierarchical'), 'Hierarchical Bayes 95% interval', 'rgba(194,24,91,1)')
        add_mean(fig, pair, view.get('hierarchical'), 'Hierarchical Bayes', 'rgba(194,24,91,1)')
    x_title = label_for('x_col', pair['x_col'])
    fig.update_layout(xaxis_title=x_title, yaxis_title=y_axis_title(pair), legend=MODEL_LEGEND, margin={'r': 20})
    with chart_output:
        chart_output.clear_output(wait=True)
        fig.show()
    with stats_output:
        stats_output.clear_output(wait=True)
        print('Zero part (logit); Positive part (OLS on y > 0).')
        display(diagnostics_table(pair))


for control in [geography, x_col, y_col, model_display, zero_values, robustness]:
    control.observe(draw_model, names='value')
display(
    widgets.VBox(
        [
            geography,
            widgets.GridBox(
                children=[x_col, y_col, model_display, zero_values],
                layout=widgets.Layout(grid_template_columns='repeat(2, minmax(220px, 1fr))', grid_gap='0.75rem'),
            ),
            chart_output,
            robustness,
            stats_output,
        ]
    )
)
draw_model()
